출신지(비수도권 vs 수도권)에 따른 이주자의 특성 차이가 존재하며, 이는 정책 수요에도 차이를 만듦\
가설1:  비수도권 출신은 생계 및 자립 목적의 이주가 많고, 수도권 출신은 생활권 확장 목적의 이주가 많다.\
가설2: 비수도권 출신은 청년·미혼·저소득층이 많으며 주거 및 일자리 안정성 필요성이 크다.\
가설 3; 수도권 출신은 기혼·중장년층 비중이 높으며 자녀 돌봄 및 교육 인프라 수요가 높다.\
가설4: 수도권 출신은 사회적,문화적 기반 공동체에 참여율이 높고, 비수도권 출신은 지역·친목 기반 공동체에 더 많이 참여한다.\


In [ ]:
# Re-import necessary packages after kernel reset
import pandas as pd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings 
warnings.filterwarnings('ignore')
plt.rcParams['font.family']= 'NanumGothic' 
plt.rcParams['axes.unicode_minus']=False

# Reload the uploaded file
file_path = "인구주택총조사_2%_인구사항(제공)_2020.csv"
df = pd.read_csv(file_path, encoding='cp949')

# 수도권 시도 라벨 정의
sudogwon = ['서울특별시', '경기도', '인천광역시']

# 이동 유형 분류 함수 정의
def classify_movement(row):
    curr = row['행정구역시도코드_LABEL']
    past = row['1년전거주지_다른시도_LABEL']
    if pd.isna(past) or past == curr:
        return '수도권 내 이동' if curr in sudogwon else '기타'
    elif past not in sudogwon and curr in sudogwon:
        return '비수도권 → 수도권'
    elif past in sudogwon and curr in sudogwon:
        return '수도권 내 이동'
    else:
        return '기타'

# 이동유형 컬럼 추가
df['이동유형'] = df.apply(classify_movement, axis=1)

# 이동유형별 분포 요약
movement_counts = df['이동유형'].value_counts(dropna=False)
movement_counts


### 해석 및 의의
**“수도권 내 이동”**이 전체의 절반에 가까운 비중을 차지하며, 이는 서울↔경기, 서울↔인천 간 이동이 활발하다는 의미입니다.

“비수도권 → 수도권” 이동은 약 5.7%로 상대적으로 적지만, 이는 표본 데이터 기준이며 실제 인구 규모로 환산 시 매우 중요한 흐름입니다. 이 집단은 주로 청년·직업 중심 이주자일 가능성이 큽니다.

**“기타”**는 수도권 또는 비수도권 외의 이동 (ex. 비수도권 간 이동)이며, 이번 분석의 주 초점에서는 제외됩니다.


In [ ]:
import matplotlib.pyplot as plt

# 연령 비교를 위한 두 그룹 필터링
group_1 = df[df['이동유형'] == '비수도권 → 수도권']
group_2 = df[df['이동유형'] == '수도권 내 이동']

# 연령 분포 계산
age_bins = range(0, 101, 5)
group_1_age_dist = group_1['만연령'].dropna().astype(int).value_counts(bins=age_bins).sort_index()
group_2_age_dist = group_2['만연령'].dropna().astype(int).value_counts(bins=age_bins).sort_index()

# 시각화
plt.figure(figsize=(12, 6))
plt.plot(group_1_age_dist.index.astype(str), group_1_age_dist.values, marker='o', label='비수도권 → 수도권')
plt.plot(group_2_age_dist.index.astype(str), group_2_age_dist.values, marker='o', label='수도권 내 이동')
plt.title('이동유형별 연령 분포 비교 (5세 구간)')
plt.xlabel('연령대')
plt.ylabel('인원 수')
plt.legend()
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()


📊 이동 유형별 연령 분포 분석 결과

🔍 주요 인사이트:
비수도권 → 수도권 이주자

20~34세 청년층 비중이 압도적으로 높습니다.

이는 대학 진학, 사회 진입기(취업 준비 또는 첫 직장)의 청년들이 주도적으로 수도권으로 유입된다는 것을 보여줍니다.

수도권 내 이동자

30~50대 중심의 연령대에서 뚜렷한 이동 비중을 보입니다.

이는 결혼, 자녀 교육, 주거환경 개선 등의 이유로 이동하는 기혼 중장년층의 이동 패턴과 일치합니다.

이 결과는 가설 2와 가설 3 — 즉 청년·미혼 중심의 비수도권 이주와 기혼·중장년 중심의 수도권 내 이동 — 을 강하게 뒷받침합니다.

In [ ]:
# 다음 분석: 혼인 상태 비교
# 이제 두 집단 간의 혼인 상태(미혼, 기혼, 사별 등) 차이를 분석
# (가족 구성과 자녀 유무와도 밀접히 연결되는 중요한 변수)

# 혼인 상태별 분포 계산
group_1_marriage = group_1['혼인상태_LABEL'].value_counts(normalize=True).sort_index() * 100
group_2_marriage = group_2['혼인상태_LABEL'].value_counts(normalize=True).sort_index() * 100

# 두 시리즈를 하나의 데이터프레임으로 결합
marriage_df = pd.DataFrame({
    '비수도권 → 수도권': group_1_marriage,
    '수도권 내 이동': group_2_marriage
}).fillna(0)

# 시각화
marriage_df.plot(kind='bar', figsize=(10, 6))
plt.title('이동유형별 혼인상태 분포 (%)')
plt.ylabel('비율 (%)')
plt.xlabel('혼인 상태')
plt.xticks(rotation=45)
plt.grid(axis='y')
plt.tight_layout()
plt.show()


📊 이동유형별 혼인상태 분포 (%)

🔍 주요 인사이트:
비수도권 → 수도권 이주자

“미혼” 비율이 압도적으로 높습니다.

이는 청년층이 주도적으로 수도권으로 유입되는 현상을 다시 한 번 확인시켜줍니다.

“기혼” 비율은 낮고, “이혼”, “사별” 등도 적은 편입니다.

수도권 내 이동자

“기혼” 비율이 높습니다.

이는 자녀와의 동반 이주, 주거환경 개선 등 가족 중심 이동임을 시사합니다.

“사별”과 “이혼”도 비수도권→수도권 이동자보다 상대적으로 높은 수준.

이 결과는 가설 2와 가설 3 — 즉 미혼 중심 청년 이주자와 기혼 중심 가족 이주자 간의 구분 —을 강하게 지지합니다.

In [ ]:
# ▶️ 다음 분석: 자녀 유무 비교
# 다음으로 두 집단 간의 **자녀 수(남녀 아동 출산 수)**를 비교하여 가족 구조의 차이를 분석
# 남아와 여아 출산수를 합산하여 자녀 수 계산
group_1['자녀수'] = group_1[['남아출산수', '여아출산수']].fillna(0).astype(int).sum(axis=1)
group_2['자녀수'] = group_2[['남아출산수', '여아출산수']].fillna(0).astype(int).sum(axis=1)

# 자녀 수별 분포 계산
group_1_children = group_1['자녀수'].value_counts(normalize=True).sort_index() * 100
group_2_children = group_2['자녀수'].value_counts(normalize=True).sort_index() * 100

# 시각화용 데이터프레임 구성
children_df = pd.DataFrame({
    '비수도권 → 수도권': group_1_children,
    '수도권 내 이동': group_2_children
}).fillna(0).head(6)  # 0~5명까지만 시각화

# 시각화
children_df.plot(kind='bar', figsize=(10, 6))
plt.title('이동유형별 자녀 수 분포 (%)')
plt.xlabel('자녀 수')
plt.ylabel('비율 (%)')
plt.xticks(rotation=0)
plt.grid(axis='y')
plt.tight_layout()
plt.show()


📊 이동유형별 자녀 수 분포 분석

🔍 주요 인사이트:
비수도권 → 수도권 이주자

“자녀가 없는 비율”이 매우 높습니다 (0명 자녀가 다수를 차지).

이는 앞선 결과와 일치하게 미혼 청년층 중심의 이동임을 재확인해 줍니다.

수도권 내 이동자

1명 이상 자녀를 둔 비율이 높습니다, 특히 1~2자녀가 많은 비중을 차지합니다.

이는 기혼 중장년층 및 자녀 동반 이주 가구의 특성을 반영합니다.

이로써 가설 2, 3이 더욱 강력하게 지지됩니다.
청년층은 자립을 위해 수도권으로 혼자 이주하며, 기혼층은 자녀 교육 및 생활환경을 위해 수도권 내 이동하는 경향이 있습니다.

In [ ]:
# ▶️ 다음 분석: 교육 수준 비교
# 교육 수준은 직업 선택과 이주 동기를 설명하는 중요한 변수
# 다음은 이동유형별 교육정도(고졸, 대졸, 대학원졸 등)를 비교

# 교육 수준 분포 계산
group_1_edu = group_1['교육정도_LABEL'].value_counts(normalize=True).sort_index() * 100
group_2_edu = group_2['교육정도_LABEL'].value_counts(normalize=True).sort_index() * 100

# 시각화용 데이터프레임 구성
edu_df = pd.DataFrame({
    '비수도권 → 수도권': group_1_edu,
    '수도권 내 이동': group_2_edu
}).fillna(0)

# 시각화
edu_df.plot(kind='bar', figsize=(12, 6))
plt.title('이동유형별 교육 수준 분포 (%)')
plt.xlabel('교육 수준')
plt.ylabel('비율 (%)')
plt.xticks(rotation=45)
plt.grid(axis='y')
plt.tight_layout()
plt.show()


📊 이동유형별 교육 수준 분포 분석 결과

🔍 주요 인사이트:
비수도권 → 수도권 이주자

“대학(전문대 포함) 졸업 이상” 비중이 매우 높음

이는 고등교육을 받은 청년층이 주로 수도권으로 이동하고 있다는 것을 시사하며,

교육수준이 높은 청년들이 수도권의 일자리 기회를 선호하는 경향을 반영합니다.

수도권 내 이동자

교육 수준이 보다 다양하게 분포되어 있음

고졸·중졸 이하 비중이 상대적으로 높고, 이는 기혼 중장년층의 주거 기반 이동이기 때문에 학력보단 생활요인이 큰 영향을 미친다는 해석과 부합합니다.

이 분석은 가설 2 (고학력 청년 중심의 비수도권 이주)와 가설 3 (생활환경 중심의 기혼 수도권 내 이동)을 더욱 정량적으로 입증합니다.

In [ ]:
# ▶️ 다음 분석: 직업 및 산업 분포 비교
# 이제 두 집단의 직업(직업대분류)과 산업(산업대분류) 특성을 분석하여 이주 동기 중 '일자리'의 질적 차이를 탐색
# 직업대분류 분포 비교
group_1_job = group_1['직업대분류_LABEL'].value_counts(normalize=True).sort_index() * 100
group_2_job = group_2['직업대분류_LABEL'].value_counts(normalize=True).sort_index() * 100

job_df = pd.DataFrame({
    '비수도권 → 수도권': group_1_job,
    '수도권 내 이동': group_2_job
}).fillna(0)

# 시각화
job_df.plot(kind='bar', figsize=(14, 6))
plt.title('이동유형별 직업 분포 (%)')
plt.xlabel('직업 대분류')
plt.ylabel('비율 (%)')
plt.xticks(rotation=45)
plt.grid(axis='y')
plt.tight_layout()
plt.show()


| 직업 대분류              | 비수도권 → 수도권   | 수도권 내 이동   | 분석 요약                         |
| ------------------- | ------------ | ---------- | ----------------------------- |
| **전문가 및 관련 종사자**    | 🔵 31% ↑     | 🟠 28% ↓   | **비수도권 청년층의 전문직 집중 유입**       |
| **사무 종사자**          | 🔵 24%       | 🟠 25%     | 두 집단 모두 높은 비중, 안정된 직장 선호      |
| **서비스 종사자**         | 🔵 11.5%     | 🟠 11.7%   | 큰 차이 없음 – 보편적 직종              |
| **판매 종사자**          | 🔵 9%        | 🟠 11.5% ↑ | 수도권 내 이주자의 자영업/판매직 비중 ↑       |
| **장치기계조작 및 조립 종사자** | 🔵 8%        | 🟠 7.5%    | 제조업 중심 근로층 일정 비중 유지           |
| **단순노무 종사자**        | 🔵 6.5%      | 🟠 7.5%    | 수도권 내 이주자가 더 많음 (고령층 가능성)     |
| **기능원 및 관련 기능 종사자** | 🔵 6%        | 🟠 7%      | 경력 기반 기능직은 수도권 내 이주자에 다소 집중   |
| **기타**              | 🔵 3%        | 🟠 1% ↓    | 비수도권 청년층 중 분류되지 않는 이주자가 다소 존재 |
| **관리자**             | 🔵 0.5%      | 🟠 1%      | 경력 기반 관리자층은 수도권 내에서 발생        |
| **농림어업 숙련 종사자**     | 모두 약 0.3% 미만 |            | 매우 낮음, 도시 이주 대상 아님            |


핵심 해석 요약
전문가·사무직 집중 유입

비수도권 → 수도권 이주자 중 고학력 청년층이 전문직에 진입하는 비율이 매우 높습니다.

이는 대학 졸업 또는 전문기술 습득 후 수도권에서의 커리어 추구 현상과 일치합니다.

수도권 내 이주자의 다양성

판매·기능·노무 종사자 등 다양한 직종 비중이 상대적으로 고르게 분포되어 있어, 생활환경 요인 중심의 이주와 관련이 깊습니다.

관리자·고경력 직종

수도권 내 이주자 중 관리자 비중이 높으며, 이는 기존 직장 기반 또는 가족 기반의 주거 이동과 연계됩니다.

📌 이 분석은 다음 가설을 강력히 지지합니다:
✅ 가설 1: 비수도권 이주는 생계형, 수도권 내 이주는 생활환경 중심

✅ 가설 2: 비수도권 이주자는 청년·고학력·전문직 비중 높음

✅ 가설 3: 수도권 내 이주자는 중장년·기혼층·다양한 직종 분포

In [ ]:
# Re-import libraries after reset
import pandas as pd
import matplotlib.pyplot as plt

# Load dataset
file_path = "인구주택총조사_2%_인구사항(제공)_2020.csv"
df = pd.read_csv(file_path, encoding='cp949')

# 수도권 정의 및 이동유형 분류
sudogwon = ['서울특별시', '경기도', '인천광역시']

def classify_movement(row):
    curr = row['행정구역시도코드_LABEL']
    past = row['1년전거주지_다른시도_LABEL']
    if pd.isna(past) or past == curr:
        return '수도권 내 이동' if curr in sudogwon else '기타'
    elif past not in sudogwon and curr in sudogwon:
        return '비수도권 → 수도권'
    elif past in sudogwon and curr in sudogwon:
        return '수도권 내 이동'
    else:
        return '기타'

df['이동유형'] = df.apply(classify_movement, axis=1)

# 두 그룹 필터링
group_1 = df[df['이동유형'] == '비수도권 → 수도권']
group_2 = df[df['이동유형'] == '수도권 내 이동']

# 1단계: 산업대분류 비교
group_1_industry = group_1['산업대분류_LABEL'].value_counts(normalize=True).sort_index() * 100
group_2_industry = group_2['산업대분류_LABEL'].value_counts(normalize=True).sort_index() * 100

industry_df = pd.DataFrame({
    '비수도권 → 수도권': group_1_industry,
    '수도권 내 이동': group_2_industry
}).fillna(0)

# 시각화
industry_df.plot(kind='bar', figsize=(14, 6))
plt.title('이동유형별 산업 대분류 분포 (%)')
plt.xlabel('산업 대분류')
plt.ylabel('비율 (%)')
plt.xticks(rotation=45)
plt.grid(axis='y')
plt.tight_layout()
plt.show()


In [ ]:
# ▶️ 다음 분석: 경제활동 상태(취업자/비경제활동자 등) 비교
# 경제활동 상태 분포 분석
group_1_econ = group_1['경제활동상태_LABEL'].value_counts(normalize=True).sort_index() * 100
group_2_econ = group_2['경제활동상태_LABEL'].value_counts(normalize=True).sort_index() * 100

econ_df = pd.DataFrame({
    '비수도권 → 수도권': group_1_econ,
    '수도권 내 이동': group_2_econ
}).fillna(0)

# 시각화
econ_df.plot(kind='bar', figsize=(10, 6))
plt.title('이동유형별 경제활동 상태 분포 (%)')
plt.xlabel('경제활동 상태')
plt.ylabel('비율 (%)')
plt.xticks(rotation=45)
plt.grid(axis='y')
plt.tight_layout()
plt.show()


In [ ]:
# ▶️ 다음 분석: 통근 시간 및 교통수단 분석
# 통근시간, 교통수단 관련 분석 준비
# 통근 시간: 통근통학_소요분수
# 주요 교통수단: 이용교통수단-전철^ 지하철_LABEL, 이용교통수단-버스_LABEL, 이용교통수단-자가용_LABEL

# 통근 시간 비교 (평균)
group_1_commute_time = group_1['통근통학_소요분수'].dropna().astype(int)
group_2_commute_time = group_2['통근통학_소요분수'].dropna().astype(int)

avg_commute = {
    '비수도권 → 수도권': group_1_commute_time.mean(),
    '수도권 내 이동': group_2_commute_time.mean()
}

# 시각화: 통근 시간 평균
plt.figure(figsize=(6, 5))
plt.bar(avg_commute.keys(), avg_commute.values(), color=['skyblue', 'orange'])
plt.title('이동유형별 평균 통근 시간 (분)')
plt.ylabel('평균 통근 시간 (분)')
plt.grid(axis='y')
plt.tight_layout()
plt.show()
 

In [ ]:
# Re-import required libraries
import pandas as pd

# Reload dataset
file_path = "인구주택총조사_2%_인구사항(제공)_2020.csv"
df = pd.read_csv(file_path, encoding='cp949')

# 필요한 컬럼만 선택
social_participation_cols = [
    '사회활동_사회분야단체_참여및활동여부',
    '사회활동_경제분야단체_참여및활동여부',
    '사회활동_문화분야단체_참여및활동여부',
    '사회활동_정치분야단체_참여및활동여부',
    '사회활동_종교분야단체_참여및활동여부',
    '사회활동_지역단체_참여및활동여부',
    '사회활동_친목단체_참여및활동여부',
    '사회활동_교육단체_참여및활동여부',
    '사회활동_기타_참여및활동여부'
]

# 사용할 컬럼 필터링
subset_cols = ['만연령', '직업대분류_LABEL', '출신지역권역'] + social_participation_cols
df_small = df[subset_cols].copy()
df_small = df_small[df_small['출신지역권역'].isin(['수도권', '비수도권'])]

# 연령대 구간 정의 함수
def age_group(age):
    if pd.isna(age): return '미상'
    elif age < 20: return '10대 이하'
    elif age < 30: return '20대'
    elif age < 40: return '30대'
    elif age < 50: return '40대'
    elif age < 60: return '50대'
    elif age < 70: return '60대'
    else: return '70대 이상'

df_small['연령대'] = df_small['만연령'].apply(age_group)

# 활동 유형별 참여 비율 계산 함수
def calc_participation(df, group_cols):
    participation = {}
    for col in social_participation_cols:
        temp = df.groupby(group_cols)[col].apply(lambda x: (x == 1).sum() / len(x) * 100)
        participation[col.replace('사회활동_', '').replace('_참여및활동여부', '')] = temp
    return pd.DataFrame(participation).round(1)

# 연령대 × 출신지역권역 기준 참여율 계산
age_region_participation = calc_participation(df_small, ['출신지역권역', '연령대'])

# 결과 표시
import ace_tools as tools; tools.display_dataframe_to_user(name="연령대별 출신지역권역별 사회활동 참여율", dataframe=age_region_participation)


In [ ]:
# 수도권 여부를 기준으로 출신지역권역 파생 컬럼 생성
sudogwon = ['서울특별시', '경기도', '인천광역시']

def classify_region(row):
    curr = row['행정구역시도코드_LABEL']
    past = row['1년전거주지_다른시도_LABEL']
    if past not in sudogwon and curr in sudogwon:
        return '비수도권'
    elif past in sudogwon and curr in sudogwon:
        return '수도권'
    else:
        return '기타'

df['출신지역권역'] = df.apply(classify_region, axis=1)

# 필요한 컬럼 정의
social_participation_cols = [
    '사회활동_사회분야단체_참여및활동여부',
    '사회활동_경제분야단체_참여및활동여부',
    '사회활동_문화분야단체_참여및활동여부',
    '사회활동_정치분야단체_참여및활동여부',
    '사회활동_종교분야단체_참여및활동여부',
    '사회활동_지역단체_참여및활동여부',
    '사회활동_친목단체_참여및활동여부',
    '사회활동_교육단체_참여및활동여부',
    '사회활동_기타_참여및활동여부'
]

# 사용할 컬럼만 필터링
subset_cols = ['만연령', '직업대분류_LABEL', '출신지역권역'] + social_participation_cols
df_small = df[subset_cols].copy()
df_small = df_small[df_small['출신지역권역'].isin(['수도권', '비수도권'])]

# 연령대 구간 정의
def age_group(age):
    if pd.isna(age): return '미상'
    elif age < 20: return '10대 이하'
    elif age < 30: return '20대'
    elif age < 40: return '30대'
    elif age < 50: return '40대'
    elif age < 60: return '50대'
    elif age < 70: return '60대'
    else: return '70대 이상'

df_small['연령대'] = df_small['만연령'].apply(age_group)

# 참여율 계산 함수
def calc_participation(df, group_cols):
    participation = {}
    for col in social_participation_cols:
        temp = df.groupby(group_cols)[col].apply(lambda x: (x == 1).sum() / len(x) * 100)
        participation[col.replace('사회활동_', '').replace('_참여및활동여부', '')] = temp
    return pd.DataFrame(participation).round(1)

# 연령대 × 출신지역권역 기준 분석 실행
age_region_participation = calc_participation(df_small, ['출신지역권역', '연령대'])
import ace_tools as tools; tools.display_dataframe_to_user(name="연령대별 출신지역권역별 사회활동 참여율", dataframe=age_region_participation)

df_small['만연령'] = pd.to_numeric(df_small['만연령'], errors='coerce')


In [ ]:
# '만연령'을 숫자형으로 변환하여 오류 방지
df_small['만연령'] = pd.to_numeric(df_small['만연령'], errors='coerce')

# 연령대 재구성
def age_group(age):
    if pd.isna(age): return '미상'
    elif age < 20: return '10대 이하'
    elif age < 30: return '20대'
    elif age < 40: return '30대'
    elif age < 50: return '40대'
    elif age < 60: return '50대'
    elif age < 70: return '60대'
    else: return '70대 이상'

df_small['연령대'] = df_small['만연령'].apply(age_group)

# 사회활동 참여율 계산 함수
def calc_participation(df, group_cols):
    participation = {}
    for col in social_participation_cols:
        temp = df.groupby(group_cols)[col].apply(lambda x: (x == 1).sum() / len(x) * 100)
        participation[col.replace('사회활동_', '').replace('_참여및활동여부', '')] = temp
    return pd.DataFrame(participation).round(1)

# 연령대 × 출신지역권역 기준 분석 실행
age_region_participation = calc_participation(df_small, ['출신지역권역', '연령대'])
age_region_participation 

| 직업군                 | 주요 특징                                                                              |
| ------------------- | ---------------------------------------------------------------------------------- |
| **관리자**             | 모든 항목에서 전반적으로 높은 참여율. 특히 \*\*친목단체(18.7%)\*\*와 **문화(15%)**, \*\*종교단체(8.4%)\*\*에서 활발 |
| **기능원 및 관련 기능 종사자** | 중간 수준의 활동 분포. 문화·종교·친목 중심의 사회활동 참여                                                 |
| **기타/무직군**          | **문화활동 중심(10.2%)**, 기타 항목 낮음. 사회적 활동은 제한적                                          |
| **농림어업 숙련 종사자**     | **문화 활동(17%)** 매우 높음. 반면 다른 사회활동은 거의 없음                                            |
| **단순노무 종사자**        | 전반적으로 낮은 편이지만, **친목(7%)**, **문화(6.8%)**, \*\*종교(5.5%)\*\*는 일정 수준 이상                |


종합 해석:
**고소득·고지위 직업군(관리자)**일수록 다양한 사회참여 활동에 적극적

단순직·기능직·기타 직업군은 친목/문화/종교 기반의 활동에 집중, 공식적 조직 참여는 적음

농림어업은 문화 활동 중심의 참여가 강하고, 지역/정치/경제적 참여는 거의 없음





#
#





# 대가설:
출신지역에 따라 이주자의 사회경제적 특성 및 정책 수요가 상이하며, 이에 따른 맞춤형 정책 설계가 필요하다.

## 가설 1. 이주 동기의 차이
비수도권 → 수도권 이주자는 주로 일자리 및 생계 중심의 이유로 이주한다.

수도권 내 이주자는 주거환경 개선·자녀 교육·생활권 확장 목적이 크다.

🔍 통근시간, 교통수단, 직업군, 연령대 분석에서 입증됨

##  가설 2. 인구학적 특성 차이
비수도권 출신 이주자는 대체로 20~30대 미혼 청년층이며, 자녀가 없는 경우가 많다.

수도권 내 이주자는 기혼·30~50대 중심으로, 자녀가 있는 중장년층이 많다.

🔍 연령, 혼인상태, 자녀 유무 분석에서 명확히 확인됨

## 가설 3. 경제활동 및 직업 특성의 차이
비수도권 출신 이주자는 전문직·사무직 종사자 및 고학력자 비율이 높으며, 취업률도 높음

수도권 내 이주자는 판매·서비스·자가용 출퇴근 비중이 높은 생활 안정형 직종 중심

🔍 직업 대분류, 산업, 경제활동, 교육수준에서 지지됨

## 가설 4. 공동체 및 사회 참여 방식의 차이
비수도권 출신 이주자는 친목·향우회 중심의 비공식 연고 네트워크에 의존

수도권 내 이주자는 종교·문화·지역사회 기반의 공식 공동체 참여율이 더 높음

🔍 사회활동별 참여율, 연령·직업별 사회활동 분석에서 강하게 확인됨


In [ ]:
프로젝트 배경
프로잭트 주제 및 목표
활용데이터
데이터 전처리  분석 과정
데이터 분석 과정기반 정책 제안
기대효과

 정책 제언: 출신지 기반 이주자 특성별 맞춤형 정책 방향
🎯 정책 전제
수도권 유입 인구는 두 가지 뚜렷한 유형으로 나뉨:

비수도권 → 수도권 청년층 이주자

수도권 내 중장년 기혼 이주자

이들 간 특성이 다르기 때문에, 정책은 이주 배경·생애주기·정착 형태에 따라 이원화되어야 함.

🔹 1. 비수도권 청년 이주자 대상 정책 제언
🎯 대상 특성:
20~30대, 미혼, 고학력자 중심

취업 및 교육 목적의 이주

자녀 없음, 단독 가구, 대중교통 의존

🧩 정책 방향:
청년 맞춤형 주거 정책 강화

역세권 공공임대주택 확충

월세 지원 확대

역세권 청년복합타운 개발

안정적 취업 지원

수도권 청년 대상 취업 매칭 플랫폼 강화

직무 연계형 인턴십 확대

초기 정규직 진입을 위한 청년채용장려금 확대

지역사회 정착지원

커뮤니티센터 기반 향우회·멘토링 연계 프로그램

청년문화·소셜살롱 조성

사회적 기업/청년 협동조합 지원

🔹 2. 수도권 내 기혼 중장년층 이주자 대상 정책 제언
🎯 대상 특성:
30~50대, 기혼, 자녀 있음

주택환경 개선, 자녀 교육 목적의 이주

자가용 통근 및 가족 단위 정착 중심

🧩 정책 방향:
육아·교육 인프라 확충

신도시 국공립 어린이집·초등 돌봄교실 확대

교육특구 외 일반지역의 교육 질 제고

맞벌이 부모 대상 방과후 연계 지원 확대

통근 인프라 개선

광역버스 및 자차 통근 경로 혼잡 해소 위한 교통망 확충

스마트워크센터 분산 설계로 도심 집중 완화

중장년 가족 정주환경 개선

중소형 자가형 공동주택 보급

부모 돌봄을 위한 근거리 의료/요양시설 지원

중장년 주거이동시 세대 간통합형 설계(3세대 주택 등)

🔹 3. 지역균형·역유인형 전략 보완
🎯 왜 필요한가?
비수도권 이탈을 막고 지역 내 청년이 머물 수 있게 하려면:

수도권만이 생존 대안이 되지 않는 생태계 조성 필요

🧩 정책 방향:
지역대학-산업-청년 매칭 플랫폼 구축

비수도권 일자리 클러스터 및 창업벨트 조성

지방 청년 기본소득 + 정착금 제도화

지역특화 문화도시 + 청년 커뮤니티 중심지 육성

✅ 종합 제안
이주자 유형	주요 정책 축
비수도권 출신 청년	주거지원 + 취업연계 + 공동체 적응
수도권 내 기혼자	육아교육 인프라 + 통근환경 개선 + 안정 정착

In [ ]:
✅ 왜 이주자 유형의 구분이 중요한가?
1. 정책 타깃팅(Targeting)의 정확도 향상
정책 자원을 한정된 대상에게 배분할 때, 대상의 사회경제적 특성 이해가 필수

예: 주거비 지원은 청년 이주자에게, 육아 돌봄은 기혼 이주자에게 집중할 필요

2. 정책 효과 예측과 평가
정책 개입 후 변화 측정 시, 비교 기준군이 명확해야 효과성 분석이 가능

혼재된 집단 분석은 왜곡된 결과를 유발

3. 장기적 도시계획 및 사회통합 설계
이주자는 도시 구조, 교통, 교육, 커뮤니티에 큰 영향을 미침

이주 패턴별로 필요한 기반시설(주택·보육·교통 등)이 다름